## Instalando Bibliotecas

In [1]:
# pip install tweepy
# pip install git+https://github.com/JustAnotherArchivist/snscrape.git
# pip install spacy
# !python -m spacy download en_core_web_sm
# !python -m spacy download pt_core_news_sm

## Importando Bibliotecas

In [1]:
from datetime import date, datetime, timedelta

import json

import snscrape.modules.twitter as sntwitter

import tweepy as tw
import requests

import pandas as pd
import spacy
import re

from datetime import date

from collections import Counter
import operator

## Getting old tweets

In [2]:
def get_historical_tweets(string_search = "", from_user = None, since = None, until = None, 
                          retweet = False, num_max_tweets = 100, lang = None):

    data_tweets = []
    
    # Definindo o período dos tweets
    search = ""
    
    if since is not None:
        search = search + f' since:{since}'    
        
    if until is not None:
        search = search + f' until:{until}' 
    
    # Definindo a inclusão ou não de retweets
    if not retweet:
        search = search + " -filter:retweets"
    
    # Definindo usuário autor do tweet
    if from_user is not None:
        search = search + f' from:{from_user}' 
        
    if lang is not None:
        search = search + f' lang:{lang}' 
        
    # Definindo pesquisa
    search = string_search + search 
    
    print(search)
    
    # Realizando pesquisa
    scraper_tweets = sntwitter.TwitterSearchScraper(search).get_items()
    
 
    
    for i, tweet in enumerate(scraper_tweets):
        if i>num_max_tweets:
            break
                    
        tweet = {"Tweet":tweet.content,
                 "Hashtags":tweet.hashtags,
                 "Mentioned_Users": [user.username for user in (tweet.mentionedUsers or [])],
#                  "Mentioned_Users": ' , '.join([user.username for user in (tweet.mentionedUsers or [])]),
                 "Outlinks": tweet.outlinks,
                 "username":tweet.user.username,
                 "user_id":tweet.user.id,
                 "is_verified":tweet.user.verified,
                 "Date":tweet.date.strftime("%d/%m/%Y, %H:%M:%S"),
#                  "coordinates":tweet.coordinates,
                 "user_location":tweet.user.location,
                 "likes":tweet.likeCount,
                 "retweets":tweet.retweetCount,
                 "language": tweet.lang
                                  
                }   
        
        data_tweets.append(tweet)
    
#     json_data_tweets = json.dumps(data_tweets)
#     json_data_tweets = json.loads(json_data_tweets)

    return data_tweets

In [10]:
#SocialMedia
search_item =  "nazismo"
from_user = None
retweet = False
since = "2022-01-01"
num_max_tweets = 30000

In [ ]:
%%time
data_tweets = get_historical_tweets(search_item, retweet = False, since = since, 
                                            num_max_tweets = num_max_tweets, lang = "pt")

nazismo since:2022-01-01 -filter:retweets lang:pt


In [ ]:
df_tweets = pd.DataFrame.from_records(data_tweets)
df_tweets.shape

In [ ]:
df_tweets_hashtag.tail()

In [ ]:
def clean_text(row):
    
    
    # Remove metion user
    row = re.sub(r'@\w+', '', row)
    
    # Remove hashtags
    row = re.sub(r'#\w+', '', row)
    
    # Remove Links
    row = re.sub(r'http\S+|www.\S+', '', row)

    return row

In [ ]:
def remove_emoji(row):
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                               u"\U00002500-\U00002BEF"  # chinese char
                               u"\U00002702-\U000027B0"
                               u"\U00002702-\U000027B0"
                               u"\U000024C2-\U0001F251"
                               u"\U0001f926-\U0001f937"
                               u"\U00010000-\U0010ffff"
                               u"\u2640-\u2642"
                               u"\u2600-\u2B55"
                               u"\u200d"
                               u"\u23cf"
                               u"\u23e9"
                               u"\u231a"
                               u"\ufe0f"  # dingbats
                               u"\u3030"
                               "]+", flags=re.UNICODE)
    
    return emoji_pattern.sub(r'', row)


In [ ]:
df_tweets["Tweet_Clean"] = df_tweets.Tweet.apply(clean_text)
df_tweets["Tweet_Clean"] = df_tweets.Tweet_Clean.apply(remove_emoji)

In [ ]:
df_tweets.to_csv("df_tweets_nazismo_2022.csv", encoding = "utf8", index = False)